<a href="https://colab.research.google.com/github/segabzz/Sistemas-de-Base-1/blob/main/Mi_copia_de_Analizador_lexico_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# GUÍA PRÁCTICA Y EVALUACIÓN ASÍNCRONA: ANALIZADORES LÉXICOS EN PYTHON Y JAVA
**Tema: Generación Automática de Analizadores Léxicos**  
**Asignatura:** Programación de Sistemas de Bas 1  

---  
## 🎯 Objetivos de Aprendizaje
1. **Comprender** la generación automática de analizadores léxicos a partir de especificaciones formales y expresiones regulares.
2. **Implementar y extender** escáneres léxicos en dos entornos tecnológicos representativos: **PLY (`ply.lex`)** en Python y **JFlex** en Java.
3. **Analizar** los mecanismos de disambiguación léxica (palabras reservadas vs. identificadores, coincidencia más larga) y el manejo de atributos/posiciones.
4. **Evaluar** estructuralmente las diferencias entre enfoques interpretados/dinámicos (PLY) y compilados/estáticos (JFlex).

---  
## 🐍 PARTE 1: Analizador Léxico en Python utilizando PLY (`ply.lex`)

### 1.1 Código Base e Instalación

In [1]:
# Instalación de la librería PLY en el entorno de Google Colab
!pip install ply > /dev/null

In [2]:
%%writefile lexer_ply.py

import ply.lex as lex

class PythonLexerPLY:
    """Especificación e implementación de un Analizador Léxico en Python usando PLY."""

    # 1. LISTA OBLIGATORIA DE TOKENS
    tokens = (
        'KEYWORD_IF',
        'KEYWORD_ELSE',
        'KEYWORD_WHILE',
        'KEYWORD_INT',
        'KEYWORD_FLOAT',
        'IDENTIFIER',
        'INTEGER_LITERAL',
        'FLOAT_LITERAL',
        'ASSIGN',
        'PLUS',
        'MINUS',
        'TIMES',
        'DIVIDE',
        'SEMICOLON',
        'LPAREN',
        'RPAREN',
        'LBRACE',
        'RBRACE',
    )

    # 2. DICCIONARIO ASOCIATIVO DE PALABRAS RESERVADAS
    reserved = {
        'if': 'KEYWORD_IF',
        'else': 'KEYWORD_ELSE',
        'while': 'KEYWORD_WHILE',
        'int': 'KEYWORD_INT',
        'float': 'KEYWORD_FLOAT'
    }

    # 3. REGLAS SIMPLES DE CADENA (t_TOKEN)
    t_ASSIGN    = r'='
    t_PLUS      = r'\+'
    t_MINUS     = r'-'
    t_TIMES     = r'\*'
    t_DIVIDE    = r'/'
    t_SEMICOLON = r';'
    t_LPAREN    = r'\('
    t_RPAREN    = r'\)'
    t_LBRACE    = r'\{'
    t_RBRACE    = r'\}'

    # Caracteres ignorados (espacios y tabuladores)
    t_ignore = ' \t'

    # 4. REGLAS BASADAS EN FUNCIONES (t_TOKEN(t))
    def t_FLOAT_LITERAL(self, t):
        r'\d+\.\d+([eE][+-]?\d+)?'
        t.value = float(t.value)
        return t

    def t_INTEGER_LITERAL(self, t):
        r'\d+'
        t.value = int(t.value)
        return t

    def t_IDENTIFIER(self, t):
        r'[a-zA-Z_][a-zA-Z0-9_]*'
        # Disambiguación entre palabras reservadas e identificadores de usuario
        t.type = self.reserved.get(t.value, 'IDENTIFIER')
        return t

    def t_newline(self, t):
        r'\n+'
        t.lexer.lineno += len(t.value)

    def t_error(self, t):
        print(f"⚠️  [Error Léxico] Línea {t.lexer.lineno}, Posición {t.lexpos}: Carácter ilegal '{t.value[0]}'")
        t.lexer.skip(1)

    def build(self, **kwargs):
        """Construye el objeto lexer dinámico mediante PLY."""
        self.lexer = lex.lex(object=self, **kwargs)

    def test(self, data: str):
        """Prueba el escáner sobre una cadena fuente imprimiendo los tokens capturados."""
        self.lexer.input(data)
        print(f"{'Línea':<8} | {'Tipo de Token':<18} | {'Lexema / Valor':<20} | {'Posición Abs.':<12}")
        print("-" * 65)
        for tok in self.lexer:
            print(f"{tok.lineno:<8} | {tok.type:<18} | {str(tok.value):<20} | {tok.lexpos:<12}")

# Práctica Base
sample_code = """
int contador = 10;
float promedio = 95.5;
while (contador) {
    promedio = promedio + 1.2e2;
    if (promedio) {
        @
    }
}
"""

#print("=== INICIANDO PRUEBA DEL ANALIZADOR LÉXICO EN PLY (Python) ===")
#py_lexer = PythonLexerPLY()
#py_lexer.build()
#py_lexer.test(sample_code)

Writing lexer_ply.py


In [4]:
from lexer_ply import PythonLexerPLY, sample_code
print("=== INICIANDO PRUEBA DEL ANALIZADOR LÉXICO EN PLY (Python) ===")
py_lexer = PythonLexerPLY()
py_lexer.build()
py_lexer.test(sample_code)

=== INICIANDO PRUEBA DEL ANALIZADOR LÉXICO EN PLY (Python) ===
Línea    | Tipo de Token      | Lexema / Valor       | Posición Abs.
-----------------------------------------------------------------
2        | KEYWORD_INT        | int                  | 1           
2        | IDENTIFIER         | contador             | 5           
2        | ASSIGN             | =                    | 14          
2        | INTEGER_LITERAL    | 10                   | 16          
2        | SEMICOLON          | ;                    | 18          
3        | KEYWORD_FLOAT      | float                | 20          
3        | IDENTIFIER         | promedio             | 26          
3        | ASSIGN             | =                    | 35          
3        | FLOAT_LITERAL      | 95.5                 | 37          
3        | SEMICOLON          | ;                    | 41          
4        | KEYWORD_WHILE      | while                | 43          
4        | LPAREN             | (                    |

### 🛠️ Desafío 1 (PLY - Python)
**Instrucciones:** Escribe a continuación la extensión de la clase `PythonLexerPLY` (o redefine la clase ampliada `PythonLexerPLYExtended`) para incluir:
1. Soporte para **cadenas literales** entrecomilladas (`"..."`) asignando el token `'STRING_LITERAL'`.
2. Soporte para **comentarios de una línea** (`//...`), configurando el lexer para que sean **ignorados en silencio** sin emitir ningún token.

In [9]:
%%writefile lexer_ply_ext.py
from lexer_ply import PythonLexerPLY

class PythonLexerPLYExtended(PythonLexerPLY):
    """Extensión del analizador léxico con soporte para cadenas y comentarios de línea."""

    tokens = PythonLexerPLY.tokens + (
        'STRING_LITERAL',
    )

    def t_STRING_LITERAL(self, t):
        r'"([^"\\\n]|\\.)*"'
        t.value = t.value[1:-1]
        return t

    def t_comment(self, t):
        r'//[^\n]*'
        pass

Writing lexer_ply_ext.py


In [10]:
import importlib
import lexer_ply_ext
importlib.reload(lexer_ply_ext)

from lexer_ply_ext import PythonLexerPLYExtended

sample_code_ext = r"""
// Declaración inicial de variables
int contador = 10;
float promedio = 95.5;
mensaje = "Hola mundo \"PLY\""; // Comentario al final de la línea
while (contador) {
    promedio = promedio / 2 + 1.2e2;
    if (promedio) {
        @
    }
}
"""

print("=== INICIANDO PRUEBA DEL ANALIZADOR LÉXICO EXTENDIDO ===")
py_lexer = PythonLexerPLYExtended()
py_lexer.build()
py_lexer.test(sample_code_ext)

=== INICIANDO PRUEBA DEL ANALIZADOR LÉXICO EXTENDIDO ===
Línea    | Tipo de Token      | Lexema / Valor       | Posición Abs.
-----------------------------------------------------------------
3        | KEYWORD_INT        | int                  | 37          
3        | IDENTIFIER         | contador             | 41          
3        | ASSIGN             | =                    | 50          
3        | INTEGER_LITERAL    | 10                   | 52          
3        | SEMICOLON          | ;                    | 54          
4        | KEYWORD_FLOAT      | float                | 56          
4        | IDENTIFIER         | promedio             | 62          
4        | ASSIGN             | =                    | 71          
4        | FLOAT_LITERAL      | 95.5                 | 73          
4        | SEMICOLON          | ;                    | 77          
5        | IDENTIFIER         | mensaje              | 79          
5        | ASSIGN             | =                    | 87   

---  
## ☕ PARTE 2: Analizador Léxico en Java utilizando JFlex

### 2.1 Configuración e Instalación del Entorno

In [11]:
import os

print("=== 1. INSTALANDO JDK EN GOOGLE COLAB ===")
ret = os.system("apt-get update -qq && apt-get install -y openjdk-11-jdk-headless > /dev/null")
print("✅ Entorno Java 11 configurado correctamente.\n" if ret == 0 else "❌ Falló la instalación del JDK.\n")

=== 1. INSTALANDO JDK EN GOOGLE COLAB ===
✅ Entorno Java 11 configurado correctamente.



# 2.2 Creación de Archivos Base de Java y Especificación JFlex

In [12]:
# --- PASO 1: DEFINICIÓN DE LA ENUMERACIÓN DE TOKENS (TokenType.java) ---
tokentype_code = """
public enum TokenType {
    // Palabras Reservadas
    KEYWORD_IF, KEYWORD_ELSE, KEYWORD_WHILE, KEYWORD_INT, KEYWORD_FLOAT,

    // Identificadores y Literales
    IDENTIFIER, INTEGER_LITERAL, FLOAT_LITERAL, STRING_LITERAL,

    // Operadores y Delimitadores
    ASSIGN, PLUS, MINUS, TIMES, DIVIDE, SEMICOLON,
    LPAREN, RPAREN, LBRACE, RBRACE,

    // Control de Errores
    ERROR
}
"""

with open("TokenType.java", "w") as f:
    f.write(tokentype_code)

# --- PASO 2: CLASE CONTENEDORA DE TOKENS CON RASTREO (Token.java) ---
token_code = """
public class Token {
    public TokenType type;
    public Object value;
    public int line;
    public int column;

    public Token(TokenType type, Object value, int line, int column) {
        this.type = type;
        this.value = value;
        this.line = line;
        this.column = column;
    }

    @Override
    public String toString() {
        return String.format("%-8d | %-8d | %-18s | %-20s", line, column, type, value);
    }
}
"""

with open("Token.java", "w") as f:
    f.write(token_code)

# --- PASO 3: ARCHIVO DE ESPECIFICACIÓN JFLEX (scanner.jflex) ---
jflex_spec_code = """
import java.io.*;

%%

/* SECCIÓN 1: OPCIONES Y DIRECTIVAS DE CONFIGURACIÓN */
%class Lexer
%public
%unicode
%line
%column
%type Token

%{
    // Métodos auxiliares para construir objetos Token con coordenadas exactas
    private Token symbol(TokenType type) {
        return new Token(type, yytext(), yyline + 1, yycolumn + 1);
    }

    private Token symbol(TokenType type, Object value) {
        return new Token(type, value, yyline + 1, yycolumn + 1);
    }
%}

/* SECCIÓN 2: MACROS Y DEFINICIONES REGULARES */
DIGITO      = [0-9]
LETRA       = [a-zA-Z_]
ID          = {LETRA}({LETRA}|{DIGITO})*
ENTERO      = {DIGITO}+
FLOTANTE    = {DIGITO}+\\.{DIGITO}+([eE][+-]?{DIGITO}+)?
WS          = [ \\t\\r\\n]+

%%

/* SECCIÓN 3: REGLAS LÉXICAS Y ACCIONES EN JAVA */

/* Ignorar espacios en blanco, tabuladores y saltos de línea */
{WS}            { /* Ignorar */ }

/* Palabras Reservadas (prioridad por ubicación superior a ID) */
"if"            { return symbol(TokenType.KEYWORD_IF); }
"else"          { return symbol(TokenType.KEYWORD_ELSE); }
"while"         { return symbol(TokenType.KEYWORD_WHILE); }
"int"           { return symbol(TokenType.KEYWORD_INT); }
"float"         { return symbol(TokenType.KEYWORD_FLOAT); }

/* Literales con conversión de tipo nativo */
{FLOTANTE}      { return symbol(TokenType.FLOAT_LITERAL, Double.parseDouble(yytext())); }
{ENTERO}        { return symbol(TokenType.INTEGER_LITERAL, Integer.parseInt(yytext())); }

/* Identificadores */
{ID}            { return symbol(TokenType.IDENTIFIER, yytext()); }

/* Operadores y Delimitadores */
"="             { return symbol(TokenType.ASSIGN); }
"+"             { return symbol(TokenType.PLUS); }
"-"             { return symbol(TokenType.MINUS); }
"*"             { return symbol(TokenType.TIMES); }
"/"             { return symbol(TokenType.DIVIDE); }
";"             { return symbol(TokenType.SEMICOLON); }
"("             { return symbol(TokenType.LPAREN); }
")"             { return symbol(TokenType.RPAREN); }
"{"             { return symbol(TokenType.LBRACE); }
"}"             { return symbol(TokenType.RBRACE); }

/* Captura de Errores Léxicos (Modo Pánico) */
.               {
                    System.err.printf("⚠️  [Error Léxico] Línea %d, Columna %d: Carácter ilegal '%s'\\n",
                                      yyline + 1, yycolumn + 1, yytext());
                    return symbol(TokenType.ERROR, yytext());
                }
"""

with open("scanner.jflex", "w") as f:
    f.write(jflex_spec_code)

# --- PASO 4: CLASE PRINCIPAL DE PRUEBA (Main.java) ---
main_code = """
import java.io.*;

public class Main {
    public static void main(String[] args) {
        if (args.length == 0) {
            System.out.println("Por favor ingrese la ruta del archivo de prueba.");
            return;
        }

        try {
            FileReader reader = new FileReader(args[0]);
            Lexer scanner = new Lexer(reader);

            System.out.println("=== ANALIZADOR LÉXICO GENERADO POR JFLEX (Java) ===");
            System.out.println(String.format("%-8s | %-8s | %-18s | %-20s", "Línea", "Columna", "Tipo de Token", "Lexema / Valor"));
            System.out.println("-".repeat(65));

            Token token;
            while ((token = scanner.yylex()) != null) {
                System.out.println(token);
            }
            System.out.println("\\n=== ESCANEO FINALIZADO EXITOSAMENTE ===");

        } catch (IOException e) {
            System.err.println("Error al leer el archivo: " + e.getMessage());
        }
    }
}
"""

with open("Main.java", "w") as f:
    f.write(main_code)

# --- PASO 5: ARCHIVO DE FUENTE DE PRUEBA (prueba.txt) ---
sample_code = """int contador = 10;
float promedio = 95.5;
while (contador) {
    promedio = promedio + 1.2e2;
    if (promedio) {
        @
    }
}
"""

with open("prueba.txt", "w") as f:
    f.write(sample_code)

print("✅ Archivos Java y especificacion scanner.jflex creados exitosamente.")

✅ Archivos Java y especificacion scanner.jflex creados exitosamente.


### 2.3 Compilación y Ejecución en Colab

In [13]:
print("=== DESCARGANDO JFLEX (JAR con dependencias) ===")
!rm -rf jflex-1.9.1 jflex.tar.gz
!wget -q https://github.com/jflex-de/jflex/releases/download/v1.9.1/jflex-1.9.1.tar.gz -O jflex.tar.gz
!tar -xzf jflex.tar.gz

print("\n=== GENERANDO Y COMPILANDO ARCHIVOS JAVA CON JFLEX ===")
!java -Xlog:disable -jar jflex-1.9.1/lib/jflex-full-1.9.1.jar scanner.jflex
!javac -J-Xlog:disable TokenType.java Token.java Lexer.java Main.java

print("\n=== SALIDA DEL ESCÁNER PROCESANDO 'prueba.txt' ===")
!java -Xlog:disable Main prueba.txt

=== DESCARGANDO JFLEX (JAR con dependencias) ===

=== GENERANDO Y COMPILANDO ARCHIVOS JAVA CON JFLEX ===
Reading "scanner.jflex"
Constructing NFA : 80 states in NFA
Converting NFA to DFA : 
........................................
42 states before minimization, 38 states in minimized DFA
Writing code to "Lexer.java"

=== SALIDA DEL ESCÁNER PROCESANDO 'prueba.txt' ===
=== ANALIZADOR LÉXICO GENERADO POR JFLEX (Java) ===
Línea    | Columna  | Tipo de Token      | Lexema / Valor      
-----------------------------------------------------------------
1        | 1        | KEYWORD_INT        | int                 
1        | 5        | IDENTIFIER         | contador            
1        | 14       | ASSIGN             | =                   
1        | 16       | INTEGER_LITERAL    | 10                  
1        | 18       | SEMICOLON          | ;                   
2        | 1        | KEYWORD_FLOAT      | float               
2        | 7        | IDENTIFIER         | promedio            


### 🛠️ Desafío 2 (JFlex - Java)
**Instrucciones:** Modifica la especificación `scanner.jflex` arriba o vuelve a escribir el archivo desde una celda agregando:
1. La definición de la regla de **cadenas literales entrecomilladas**: `\"[^\"]*\" { return symbol(TokenType.STRING_LITERAL, yytext()); }`
2. La regla para **comentarios de una línea**: `"//".* { /* Ignorar */ }`
Recompila con `!jflex scanner.jflex && javac *.java` y ejecuta sobre un archivo de prueba ampliado.

In [14]:
%%writefile scanner.jflex
import java.io.*;

%%

/* SECCIÓN 1: OPCIONES Y DIRECTIVAS DE CONFIGURACIÓN */
%class Lexer
%public
%unicode
%line
%column
%type Token

%{
    private Token symbol(TokenType type) {
        return new Token(type, yytext(), yyline + 1, yycolumn + 1);
    }

    private Token symbol(TokenType type, Object value) {
        return new Token(type, value, yyline + 1, yycolumn + 1);
    }
%}

/* SECCIÓN 2: MACROS Y DEFINICIONES REGULARES */
DIGITO      = [0-9]
LETRA       = [a-zA-Z_]
ID          = {LETRA}({LETRA}|{DIGITO})*
ENTERO      = {DIGITO}+
FLOTANTE    = {DIGITO}+\.{DIGITO}+([eE][+-]?{DIGITO}+)?
WS          = [ \t\r\n]+

%%

/* SECCIÓN 3: REGLAS LÉXICAS Y ACCIONES EN JAVA */

/* Ignorar espacios en blanco, tabuladores y saltos de línea */
{WS}            { /* Ignorar */ }

/* Comentarios de una línea (//...) ignorados en silencio */
"//".*          { /* Ignorar */ }

/* Palabras Reservadas */
"if"            { return symbol(TokenType.KEYWORD_IF); }
"else"          { return symbol(TokenType.KEYWORD_ELSE); }
"while"         { return symbol(TokenType.KEYWORD_WHILE); }
"int"           { return symbol(TokenType.KEYWORD_INT); }
"float"         { return symbol(TokenType.KEYWORD_FLOAT); }

/* Literales con conversión de tipo nativo */
{FLOTANTE}      { return symbol(TokenType.FLOAT_LITERAL, Double.parseDouble(yytext())); }
{ENTERO}        { return symbol(TokenType.INTEGER_LITERAL, Integer.parseInt(yytext())); }
\"[^\"]*\"      { return symbol(TokenType.STRING_LITERAL, yytext()); }

/* Identificadores */
{ID}            { return symbol(TokenType.IDENTIFIER, yytext()); }

/* Operadores y Delimitadores */
"="             { return symbol(TokenType.ASSIGN); }
"+"             { return symbol(TokenType.PLUS); }
"-"             { return symbol(TokenType.MINUS); }
"*"             { return symbol(TokenType.TIMES); }
"/"             { return symbol(TokenType.DIVIDE); }
";"             { return symbol(TokenType.SEMICOLON); }
"("             { return symbol(TokenType.LPAREN); }
")"             { return symbol(TokenType.RPAREN); }
"{"             { return symbol(TokenType.LBRACE); }
"}"             { return symbol(TokenType.RBRACE); }

/* Captura de Errores Léxicos (Modo Pánico) */
.               {
                    System.err.printf("⚠️  [Error Léxico] Línea %d, Columna %d: Carácter ilegal
                    '%s'\n",
                                      yyline + 1, yycolumn + 1, yytext());
                    return symbol(TokenType.ERROR, yytext());
                }

Overwriting scanner.jflex


In [16]:
# 1. Crear archivo de prueba ampliado con cadenas y comentarios
prueba_ext = """// Prueba ampliada del analizador léxico en JFlex
int contador = 10;
float promedio = 95.5;
mensaje = "Hola mundo desde JFlex"; // Comentario al final de la línea
while (contador) {
    promedio = promedio / 2 + 1.2e2;
    if (promedio) {
        aviso = "Calculo terminado";
        @
    }
}
"""

with open("prueba_ext.txt", "w") as f:
    f.write(prueba_ext)

# 2. Recompilar scanner.jflex y archivos .java
print("=== RECOMPILANDO JFLEX Y JAVA ===")
!java -Xlog:disable -jar jflex-1.9.1/lib/jflex-full-1.9.1.jar scanner.jflex && javac-J-Xlog:disable *.java

# 3. Ejecutar el escáner sobre el archivo de prueba ampliado
print("\n=== SALIDA DEL ESCÁNER PROCESANDO 'prueba_ext.txt' ===")
!java -Xlog:disable Main prueba_ext.txt

=== RECOMPILANDO JFLEX Y JAVA ===
Reading "scanner.jflex"

Error in file "scanner.jflex" (line 70): 
Unexpected character
                    System.err.printf("⚠️  [Error Léxico] Línea %d, Columna %d: Carácter ilegal
                                      ^
1 error, 0 warnings.

=== SALIDA DEL ESCÁNER PROCESANDO 'prueba_ext.txt' ===
=== ANALIZADOR LÉXICO GENERADO POR JFLEX (Java) ===
Línea    | Columna  | Tipo de Token      | Lexema / Valor      
-----------------------------------------------------------------
1        | 1        | DIVIDE             | /                   
1        | 2        | DIVIDE             | /                   
1        | 4        | IDENTIFIER         | Prueba              
1        | 11       | IDENTIFIER         | ampliada            
1        | 20       | IDENTIFIER         | del                 
1        | 24       | IDENTIFIER         | analizador          
1        | 35       | IDENTIFIER         | l                   
⚠️  [Error Léxico] Línea 1, Columna

---  
## 📝 BITÁCORA Y REPORTE DE LABORATORIO

**Nombre del Estudiante:** Zertuche Zamora Sergio Gabriel

**Matrícula / Código:** 2171390053  

**Fecha:** 16/11/2026  

---
### Sección 1: Verificación de Coincidencia y Resolución de Ambigüedades
1. **Mecanismo de la regla más larga (*Longest Match*):**  
   Al analizar la subcadena `1.2e2`, explica paso a paso por qué el autómata generado no la separa como el entero `1`, el operador `.` y el identificador/entero `2e2`, sino como un único token `FLOAT_LITERAL`.  
   *Respuesta del alumno:*
   >El analizador léxico aplica el principio de coincidencia más larga (Longest Match / Maximal
  Munch), por lo que el autómata (AFD) sigue consumiendo caracteres mientras exista una transición
  válida, en lugar de detenerse en el primer estado de aceptación:
>  1. Al leer 1: Alcanza un estado de aceptación para INTEGER_LITERAL, pero no se detiene porque el carácter siguiente (.) permite continuar hacia la regla FLOAT_LITERAL.
    2. Al leer . y 2: Forma 1.2 (estado de aceptación de FLOAT_LITERAL), pero continúa porque el
  patrón admite la parte exponencial opcional ([eE][+-]?[0-9]+)?.
    3. Al leer e y 2: Completa el lexema 1.2e2 (5 caracteres) dentro de FLOAT_LITERAL.
    4. Al encontrar el delimitador siguiente (por ejemplo ; o espacio): Ya no hay transición válida;
  el autómata se detiene y emite el token correspondiente al prefijo válido más largo encontrado,
  que es 1.2e2 como un único FLOAT_LITERAL (descartando el corte previo en 1).

2. **Disambiguación de Palabras Reservadas vs. Identificadores:**  
   * En **JFlex**, ¿por qué la posición (física) de las reglas de `"if"`, `"while"`, etc., por encima de la regla `{ID}` garantiza que no sean clasificadas como variables?
   * En **PLY**, ¿por qué se utiliza el método `reserved.get(t.value, 'IDENTIFIER')` dentro de la función `t_IDENTIFIER` en lugar de crear una regla independiente `t_KEYWORD_IF = r'if'`?  
   *Respuesta del alumno:*
   >En JFlex (orden de reglas): Cuando dos reglas coinciden con un lexema de la misma longitud (por
  ejemplo, "if" coincide tanto con "if" como con {ID} con 2 caracteres), JFlex desempata aplicando
  la regla que aparece primero en el archivo. Al colocar "if", "while", etc. por encima de {ID},
  siempre ganan el empate y se evita que se clasifiquen como identificadores (mientras que nombres
  más largos como if_temp siguen cayendo en {ID} por Longest Match).
  
      >En PLY (reserved.get(...) dentro de t_IDENTIFIER):
      >1. Prioridad y prefijos: En PLY, las reglas basadas en función (def t_IDENTIFIER) siempre
          tienen mayor prioridad que las reglas de cadena (t_KEYWORD_IF = r'if'), por lo que una regla
          simple sería ignorada o, si se pusiera como función antes que t_IDENTIFIER, podría partir
          erróneamente identificadores que comienzan con una palabra reservada (ej. leer iffy como
          KEYWORD_IF "if" + IDENTIFIER "fy").
       2. Eficiencia y limpieza: Capturar primero la palabra completa con t_IDENTIFIER y luego
          consultar el diccionario reserved en O(1) resuelve la ambigüedad de forma exacta sin saturar
          la expresión regular principal con decenas de reglas individuales.
      
---
### Sección 2: Conversión y Tipado de Atributos Léxicos
1. **Conversión de Tipos:**  
   En la regla `t_FLOAT_LITERAL` de PLY se ejecuta `t.value = float(t.value)`, y en JFlex se pasa `Double.parseDouble(yytext())`. ¿Por qué es fundamental que esta conversión ocurra dentro del analizador léxico antes de entregar el token al analizador sintáctico/semántico?  
   *Respuesta del alumno:*
   >Porque garantiza la separación de responsabilidades entre fases del compilador: el analizador
  léxico ya validó la estructura del lexema (por ejemplo, 1.2e2) y lo transforma una sola vez en su
  tipo de dato nativo (float / Double). De este modo, los analizadores sintáctico y semántico
  reciben un atributo listo para operar aritméticamente, verificar tipos o evaluar expresiones
  constantes (constant folding) sin tener que volver a parsear cadenas de texto ni manejar errores
  de formato en etapas posteriores.

---
### Sección 3: Análisis Estructural y Manejo de Posiciones
1. **Comparación de Rastreos de Coordenadas:**  
   Explica la diferencia técnica entre cómo rastrea PLY la ubicación de los tokens (utilizando `t.lexer.lineno` y `t.lexpos`) y cómo lo realiza JFlex mediante las directivas `%line` y `%column` con las variables internas `yyline` y `yycolumn`.  
   *Respuesta del alumno:*
   >En PLY (manual y posición absoluta):
      • t.lexer.lineno inicia en 1, pero PLY no lo actualiza automáticamente; el programador debe
      definir t_newline para incrementarlo (t.lexer.lineno += len(t.value)).
      • t.lexpos no indica la columna de la línea, sino el índice absoluto (base 0) del carácter
      desde el inicio de toda la cadena de entrada (por lo que, si se requiere la columna exacta,
      debe calcularse manualmente respecto al último \n).
      
      >En JFlex (automático y coordenadas línea/columna relativas):
      Las directivas %line y %column generan código automáticamente en el escáner para mantener
      yyline (número de línea) y yycolumn (columna relativa dentro de la línea actual, que se
      reinicia en cada salto de línea).
      • Ambas variables en JFlex son base 0, por lo que en scanner.jflex se les suma 1 (yyline + 1,
      yycolumn + 1) al construir cada Token.

---
### Sección 4: Extensión de Reglas (Evidencia del Desafío)
Pega las líneas de código o expresiones regulares que agregaste en PLY y JFlex para dar soporte a cadenas literales y comentarios de una línea:

**Código/Reglas agregadas en PLY (Python):**

```python
# Pegar aquí el fragmento de código
class PythonLexerPLYExtended(PythonLexerPLY):
# 1. Extensión de la lista de tokens
tokens = PythonLexerPLY.tokens + (
    'STRING_LITERAL',
)

# 2. Regla para cadenas literales entrecomilladas ("...")
def t_STRING_LITERAL(self, t):
    r'"([^"\\\n]|\\.)*"'
    t.value = t.value[1:-1]
    return t

# 3. Regla para comentarios de una línea (//...) ignorados en silencio
def t_comment(self, t):
    r'//[^\n]*'
    pass
```

**Reglas agregadas en scanner.jflex (Java):**

```jflex
/* Pegar aquí las reglas de JFlex */

/* Comentarios de una línea (//...) ignorados en silencio */
"//".*          { /* Ignorar */ }

/* Cadenas literales entrecomilladas ("...") */
\"[^\"]*\"      { return symbol(TokenType.STRING_LITERAL, yytext()); }
```

---
### Sección 5: Cuadro Comparativo Arquitectónico

| Criterio de Comparación | PLY (Python) | JFlex (Java) |
| :--- | :--- | :--- |
| **Naturaleza del Generador** | (Interpretado / Reflexivo / Compilado) | (Interpretado / Reflexivo / Compilado) |
| **Definición de Patrones** | Docstrings en funciones Python | Archivo de especificación `.jflex` |
| **Manejo de Palabras Reservadas** | Diccionario asociativo `reserved`consultado dentro de `t_IDENTIFIER` mediante `self.reserved.get(t.value, 'IDENTIFIER')`. | Reglas literales (`"if"`, `"while"`, etc.) colocadas físicamente por encima de `{ID}` para ganar empates de longitud. |
| **Rastreo de Línea y Columna** | **Manual y absoluto**: requiere definir `t_newline` para actualizar `t.lexer.lineno`, y `t.lexpos` guarda el índice absoluto (base 0) sin columna automática. | **Automático y relativo**: las directivas `%line` y `%column` actualizan `yyline` y `yycolumn` (base 0, relativos a la línea actual). |
| **Recuperación de Errores** | Función especial `t_error(self, t)` que notifica el carácter ilegal y avanza 1 carácter con `t.lexer.skip(1)` (modo pánico). | Regla comodín `.` al final de las reglas que reporta el error con `yyline`/`yycolumn` y devuelve `TokenType.ERROR` (modo pánico). |

---  
**Conclusión General del Experimento:**  
> Tanto PLY (Python) como JFlex (Java) cumplen eficazmente con la fase de análisis léxico, pero
  responden a necesidades distintas en el desarrollo de un compilador real:

  >• PLY (Python) destaca por su rapidez de prototipado y simplicidad: permite definir tokens,
  conversiones de tipos y reglas de error directamente en Python sin pasos de compilación
  intermedios, lo cual es ideal para proyectos académicos, lenguajes de dominio específico (DSL) o
  pruebas rápidas. Sin embargo, al depender de reflexión y del motor de expresiones regulares de
  Python en tiempo de ejecución, resulta menos eficiente para procesar grandes volúmenes de código
  fuente y carece de rastreo automático de columnas y verificación estática de tipos.
  
  >• JFlex (Java) sobresale en rendimiento, robustez y escalabilidad: genera un autómata finito
  determinista (AFD) minimizado y compilado a código nativo de la JVM, ofrece rastreo automático de
  línea y columna (%line, %column), tipado fuerte en los tokens (TokenType) y una resolución clara
  de ambigüedades por coincidencia más larga y orden de reglas. Su principal desventaja es una
  mayor curva de configuración y un flujo de desarrollo más rígido al requerir generación previa y
  recompilación tras cada cambio en la especificación .jflex.

  >En resumen, PLY es la mejor opción para prototipar e iterar con agilidad, mientras que JFlex
  resulta más adecuado para compiladores de producción donde el rendimiento del escáner, la
  precisión de coordenadas y la robustez del tipado son prioritarios.